# 1 — Build a causal text transformer from scratch

**Learning goal:** train a small decoder-only transformer to predict the next
character in Tiny Shakespeare, understand every important tensor, and generate a
sample. This is a learning model, not a production LLM.

We use PyTorch's low-level building blocks and write the training loop ourselves.
The answer key at the bottom is a complete runnable path. Start with `FAST_MODE =
True`; correctness and a downward trend matter more than literary output.

> A *model* is a parameterized function. *Training* adjusts its parameters to make
> its predictions less wrong on examples. A transformer is a neural-network family
> whose key operation, attention, lets each sequence position combine information
> from other positions.


## Map of the pipeline

`text → tokens → context/target batches → embeddings → transformer blocks → logits
→ cross-entropy loss → gradients → AdamW update → validation → generation`

Three splits have different jobs:

- **train:** gradients update model parameters;
- **validation:** choose hyperparameters and detect overfitting;
- **test:** one final, relatively unbiased report—do not repeatedly tune on it.

We split contiguous text chronologically. Randomly splitting individual overlapping
windows can leak almost-identical text across splits.


In [ ]:
# Imports, reproducibility, and hardware selection
from pathlib import Path
from urllib.request import urlretrieve
import math, random, time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
FAST_MODE = True
print("PyTorch:", torch.__version__, "| device:", device, "| fast mode:", FAST_MODE)


### Python bridge for R users

- Python indexing starts at 0. `x[:10]` means the first ten values; the upper bound
  is excluded. It resembles R's `head(x, 10)`, but not R's 1-based indexes.
- A dictionary such as `{"a": 0}` is a named lookup table, similar to a named R
  vector/list. `d[k]` looks up key `k`.
- `[f(x) for x in items]` is a list comprehension, much like `lapply(items, f)`.
- Tensor shapes are written `(B, T, C)`: batch, time/sequence, channels/features.
- A Python `class` bundles parameters and behavior. PyTorch models subclass
  `nn.Module`; `forward` defines the computation.
- `@torch.no_grad()` is a decorator: it changes how the function below runs, here
  disabling gradient bookkeeping during evaluation.


## 1. Data and tokenization

Tiny Shakespeare is about 1.1 MB of text. A **token** is one discrete unit presented
to the model. Modern LLMs usually use subword tokens; here each distinct character
is a token. Character tokenization is transparent and needs only ~65 vocabulary
entries, but sequences are longer and semantic units such as words are not explicit.

Tokenization maps text to integer IDs. IDs are labels, not magnitudes: token 40 is
not “twice” token 20. An embedding table later learns a vector for every ID.

Dataset: [Karpathy's Tiny Shakespeare](https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt)  
API: [`torch.tensor`](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
data_dir = Path("data"); data_dir.mkdir(exist_ok=True)
data_path = data_dir / "tiny_shakespeare.txt"
if not data_path.exists():
    urlretrieve(DATA_URL, data_path)
text = data_path.read_text(encoding="utf-8")
print(f"{len(text):,} characters | preview:\n{text[:300]}")


### Exercise — build a character vocabulary

Create a sorted list of unique characters and two dictionaries: character to integer and integer to character. Sorting makes IDs reproducible.

Replace the `None`/`TODO` portion below. The next cell is a small unit test: green
output means the behavior and important shapes are correct, not that there is only
one valid solution.


In [ ]:
chars = None  # TODO: sorted(set(text))
stoi = None   # TODO: {character: integer, ...}
itos = None   # TODO: {integer: character, ...}


In [ ]:
if chars is None:
    print("🟡 Not attempted yet. Hint: enumerate(chars) yields (index, value).")
else:
    assert len(chars) == len(set(text)) and chars == sorted(chars)
    assert all(itos[stoi[ch]] == ch for ch in chars)
    print(f"🟢 Vocabulary is reversible and has {len(chars)} tokens.")


## 2. Encode, decode, and split

Encoding is a deterministic data transformation, not something learned. We turn
the entire corpus into a one-dimensional `torch.long` tensor because embedding
layers require integer indexes. Decoding reverses that operation for inspection.

The 80/10/10 split is a convention, not a law. The key rule is that test data stays
untouched until the end. For time-ordered or authored data, preserve order unless
you have a reason not to.


### Exercise — write encode and decode

Use the vocabulary dictionaries. A function is introduced by `def`; its indented body ends when indentation ends.

Replace the `None`/`TODO` portion below. The next cell is a small unit test: green
output means the behavior and important shapes are correct, not that there is only
one valid solution.


In [ ]:
def encode(s):
    return None  # TODO: a list of IDs

def decode(ids):
    return None  # TODO: one string


In [ ]:
if encode("Hi") is None:
    print("🟡 Not attempted yet. Hint: ''.join(...) combines characters.")
else:
    probe = text[:100]
    assert decode(encode(probe)) == probe
    print("🟢 Round trip passed:", repr(decode(encode("ROMEO"))))


### Exercise — make train/validation/test tensors

Encode the text once, then take contiguous 80%, 10%, and 10% slices. Use dtype `torch.long`.

Replace the `None`/`TODO` portion below. The next cell is a small unit test: green
output means the behavior and important shapes are correct, not that there is only
one valid solution.


In [ ]:
data = None
train_data = None
val_data = None
test_data = None


In [ ]:
if data is None:
    print("🟡 Not attempted. Hint: n = len(data); data[:int(.8*n)] is train.")
else:
    assert data.dtype == torch.long
    assert len(train_data)+len(val_data)+len(test_data) == len(data)
    assert torch.equal(torch.cat([train_data,val_data,test_data]), data)
    print("🟢 Split sizes:", *(len(x) for x in (train_data,val_data,test_data)))


## 3. Context windows and shifted labels

Language modeling uses self-supervision: the raw text supplies its own labels. If a
window is `Hell`, its targets are `ello`. At every position, the model predicts the
next token. A context length `T` yields `T` supervised predictions per example.

`batch_size` (`B`) is the number of windows processed before one optimizer update.
`block_size` (`T`) is the maximum context length. Larger values cost more memory;
self-attention work grows roughly with `T²`.

We randomly sample windows *inside one already-separated split*. This is stochastic
gradient descent: each update sees a small estimate of the full-data gradient.


In [ ]:
# Temporary reference encoding so later exercises can run independently.
_chars = sorted(set(text)); _stoi = {ch:i for i,ch in enumerate(_chars)}
_data = torch.tensor([_stoi[ch] for ch in text], dtype=torch.long)
_n = len(_data); _train = _data[:int(.8*_n)]; _val = _data[int(.8*_n):int(.9*_n)]
batch_size = 16 if FAST_MODE else 32
block_size = 64 if FAST_MODE else 128


### Exercise — sample a shifted minibatch

Implement `get_batch(source)`. Sample `batch_size` legal start indexes, stack input windows and their one-character-shifted targets, then move both to `device`.

Replace the `None`/`TODO` portion below. The next cell is a small unit test: green
output means the behavior and important shapes are correct, not that there is only
one valid solution.


In [ ]:
def get_batch(source):
    # TODO
    return None, None


In [ ]:
xb, yb = get_batch(_train)
if xb is None:
    print("🟡 Not attempted. Useful APIs: torch.randint, torch.stack.")
else:
    assert xb.shape == yb.shape == (batch_size, block_size)
    assert torch.equal(xb[:,1:], yb[:,:-1])
    assert xb.device == device
    print("🟢 Batch shape:", tuple(xb.shape), "| first pair:", xb[0,:5].tolist(), yb[0,:5].tolist())


## 4. What attention computes

For every token representation `x`, learned linear layers create a **query** (what
this position seeks), **key** (what this position advertises), and **value** (the
information it can contribute). Similarity scores are `Q @ Kᵀ / sqrt(head_dim)`.
Softmax turns scores into nonnegative weights summing to one, then weights mix `V`.

Multiple heads repeat this with different learned projections, allowing different
relationships. The outputs are concatenated and projected. Positional embeddings
are essential because plain attention has no inherent word order.

For next-token prediction, position `t` must not inspect tokens after `t`; otherwise
training leaks the answer. A **causal mask** sets future attention scores to negative
infinity before softmax. Regression in Notebook 2 intentionally omits this mask.

Reference: [Attention Is All You Need](https://arxiv.org/abs/1706.03762),
[`nn.MultiheadAttention`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html)


In [ ]:
# Visualize a causal mask: 0 = visible, -inf = forbidden.
demo_mask = torch.triu(torch.full((8, 8), float("-inf")), diagonal=1)
plt.figure(figsize=(4, 3)); plt.imshow(torch.isfinite(demo_mask), cmap="Blues")
plt.xlabel("key position"); plt.ylabel("query position"); plt.title("Causal visibility")
plt.colorbar(label="can attend"); plt.show()


## 5. Architecture and parameters

- `vocab_size`: number of possible character classes.
- `d_model`: width of each token representation; larger means more capacity/cost.
- `n_head`: parallel attention heads; `d_model` must divide evenly by this.
- `n_layer`: transformer blocks stacked in depth.
- `dim_feedforward`: hidden width of each block's position-wise MLP.
- `dropout`: randomly zeros activations only during training to reduce overfitting.
- **Residual connections** add a block's input to its output, aiding gradient flow.
- **Layer normalization** stabilizes each token's feature scale.
- The final linear head maps `d_model` features to one logit per vocabulary token.

A **logit** is an unrestricted score, not a probability. Cross-entropy internally
applies log-softmax and rewards a high score for the correct next token. Random
guess loss is `ln(vocab_size)` and perplexity is `exp(loss)`.

Reference: [`TransformerEncoderLayer`](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html),
[`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)


### Exercise — define the language model

Complete the model using token + position embeddings, `TransformerEncoder` with `batch_first=True`, a causal mask, final layer norm, and vocabulary projection.

Replace the `None`/`TODO` portion below. The next cell is a small unit test: green
output means the behavior and important shapes are correct, not that there is only
one valid solution.


In [ ]:
class TinyCausalTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=96, n_head=4, n_layer=2, dropout=.1, max_len=128):
        super().__init__()
        # TODO: define layers

    def forward(self, idx, targets=None):
        # TODO: return logits shaped (B,T,V), and optional scalar loss
        return None, None


In [ ]:
try:
    candidate = TinyCausalTransformer(len(_chars), max_len=block_size).to(device)
    logits, loss = candidate(xb, yb)
    assert logits.shape == (batch_size, block_size, len(_chars))
    assert loss.ndim == 0 and torch.isfinite(loss)
    print("🟢 Forward pass works; initial loss:", round(loss.item(), 3))
except Exception as exc:
    print("🟡 Finish the model, then rerun. Current issue:", type(exc).__name__, exc)


## 6. Training and evaluation

One optimizer step is: clear old gradients → forward pass → loss → backward pass →
optionally clip gradients → update parameters. `AdamW` adapts each parameter's step
size and decouples weight decay. The **learning rate** is often the most important
hyperparameter: too high diverges, too low crawls.

Evaluation uses `model.eval()` (disables dropout), `torch.no_grad()` (saves memory),
and multiple batches (less noisy). Never call `backward()` or `optimizer.step()` on
validation/test data.

Reference: [`AdamW`](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html),
[`clip_grad_norm_`](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html),
[TensorBoard](https://docs.pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html)


### Exercise — write one optimizer step

Given `model`, `optimizer`, `xb`, and `yb`, write the five core lines. Clip total gradient norm to 1.0 before stepping.

Replace the `None`/`TODO` portion below. The next cell is a small unit test: green
output means the behavior and important shapes are correct, not that there is only
one valid solution.


In [ ]:
# optimizer.zero_grad(set_to_none=True)
# TODO: forward, backward, clip, step


In [ ]:
print("Self-check questions: Did zero_grad come before backward? Did optimizer.step come last?")


## 7. Generation is repeated classification

Generation feeds the current context through the model, selects from the final
position's next-token distribution, appends that token, and repeats. **Temperature**
divides logits: below 1 is safer/sharper; above 1 is more random. **Top-k** sampling
keeps only the k highest-scoring choices. Greedy argmax can become repetitive.

Training and generation differ: training predicts all positions in parallel under
a causal mask; generation creates one new position at a time.


# Answer key — complete runnable implementation

Try the exercises first. This section deliberately uses names prefixed with `ans_`
so it does not depend on incomplete exercise cells. Running from here trains a fresh
small model, logs metrics, saves the best checkpoint, evaluates once on test data,
and generates a sample.


In [ ]:
# Answer 1–3: vocabulary, encoding, and leakage-safe split
ans_chars = sorted(set(text))
ans_stoi = {ch: i for i, ch in enumerate(ans_chars)}
ans_itos = {i: ch for i, ch in enumerate(ans_chars)}
ans_encode = lambda s: [ans_stoi[ch] for ch in s]
ans_decode = lambda ids: "".join(ans_itos[int(i)] for i in ids)
ans_data = torch.tensor(ans_encode(text), dtype=torch.long)
n = len(ans_data)
ans_train = ans_data[:int(.8*n)]
ans_val = ans_data[int(.8*n):int(.9*n)]
ans_test = ans_data[int(.9*n):]
print(len(ans_chars), "tokens; random baseline loss", round(math.log(len(ans_chars)), 3))


In [ ]:
# Answer 4: random contiguous batches
def ans_get_batch(source):
    starts = torch.randint(0, len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i:i+block_size] for i in starts])
    y = torch.stack([source[i+1:i+block_size+1] for i in starts])
    return x.to(device), y.to(device)

ax, ay = ans_get_batch(ans_train)
assert ax.shape == ay.shape == (batch_size, block_size)


In [ ]:
# Answer 5: causal transformer
class AnswerCausalTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=96, n_head=4, n_layer=2, dropout=.1, max_len=128):
        super().__init__()
        self.max_len = max_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_head, 4*d_model, dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, n_layer, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.max_len: raise ValueError(f"T={T} exceeds max_len={self.max_len}")
        pos = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(pos)[None, :, :]
        mask = torch.triu(torch.full((T, T), float("-inf"), device=idx.device), diagonal=1)
        x = self.blocks(x, mask=mask)
        logits = self.lm_head(self.norm(x))
        loss = None if targets is None else F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, new_tokens, temperature=0.8, top_k=20):
        self.eval()
        for _ in range(new_tokens):
            logits, _ = self(idx[:, -self.max_len:])
            logits = logits[:, -1, :] / temperature
            if top_k:
                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < values[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, 1)), dim=1)
        return idx

ans_model = AnswerCausalTransformer(len(ans_chars), max_len=block_size).to(device)
print(f"Parameters: {sum(p.numel() for p in ans_model.parameters()):,}")


In [ ]:
# Answer 6: evaluation and explicit training loop
@torch.no_grad()
def estimate_loss(model, source, batches=5):
    model.eval(); values = []
    for _ in range(batches):
        x, y = ans_get_batch(source); _, loss = model(x, y); values.append(loss.item())
    model.train()
    return float(np.mean(values))

steps = 80 if FAST_MODE else 1000
eval_every = 20 if FAST_MODE else 100
optimizer = torch.optim.AdamW(ans_model.parameters(), lr=3e-4, weight_decay=.01)
writer = SummaryWriter("runs/shakespeare")
Path("checkpoints").mkdir(exist_ok=True)
history = {"step": [], "train": [], "val": []}; best_val = float("inf")

for step in range(steps + 1):
    if step % eval_every == 0:
        tr = estimate_loss(ans_model, ans_train); va = estimate_loss(ans_model, ans_val)
        history["step"].append(step); history["train"].append(tr); history["val"].append(va)
        writer.add_scalars("loss", {"train": tr, "validation": va}, step)
        print(f"step {step:4d} | train {tr:.3f} | val {va:.3f} | ppl {math.exp(va):.1f}")
        if va < best_val:
            best_val = va; torch.save(ans_model.state_dict(), "checkpoints/shakespeare_best.pt")
    if step == steps: break
    xb, yb = ans_get_batch(ans_train)
    optimizer.zero_grad(set_to_none=True)
    _, loss = ans_model(xb, yb)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(ans_model.parameters(), 1.0)
    optimizer.step()
    writer.add_scalar("train/grad_norm", float(grad_norm), step)
writer.close()


In [ ]:
# Learning curves and final held-out test evaluation
plt.plot(history["step"], history["train"], marker="o", label="train")
plt.plot(history["step"], history["val"], marker="o", label="validation")
plt.axhline(math.log(len(ans_chars)), color="gray", ls="--", label="random baseline")
plt.xlabel("optimizer step"); plt.ylabel("cross-entropy"); plt.legend(); plt.show()
ans_model.load_state_dict(torch.load("checkpoints/shakespeare_best.pt", map_location=device, weights_only=True))
test_loss = estimate_loss(ans_model, ans_test, batches=10)
print(f"Test loss {test_loss:.3f}; perplexity {math.exp(test_loss):.1f}")


In [ ]:
# Generate. FAST_MODE learns formatting before language; longer training is more legible.
prompt = "ROMEO:\n"
context = torch.tensor([ans_encode(prompt)], dtype=torch.long, device=device)
sample = ans_model.generate(context, new_tokens=300, temperature=.8, top_k=20)
print(ans_decode(sample[0].tolist()))


In [ ]:
# Optional TensorBoard inside Jupyter/Colab
%load_ext tensorboard
%tensorboard --logdir runs/shakespeare


## Interpret, debug, and iterate

Healthy signs: training and validation loss both fall; validation may be noisy. If
training falls while validation rises, the model is overfitting. If neither falls,
check shifted targets and masks first, then try learning rate/capacity/steps. NaNs
often indicate an excessive learning rate or unstable values.

Suggested controlled experiments (change one thing and record train/validation):

1. Remove positional embeddings. What ordering ability remains?
2. Temporarily remove the causal mask. Why can validation loss look deceptively good?
3. Compare greedy, temperature 0.5/1.2, and top-k sampling.
4. Double context length; note both quality and time/memory.
5. Compare character tokens with a subword tokenizer conceptually.

Production LLMs add enormous datasets/models, sophisticated tokenizers, distributed
training, schedules, mixed precision, and alignment—but the central next-token
objective and attention mechanism here are genuine.


## References and next reading

- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- PyTorch [`nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)
- PyTorch [`Embedding`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)
- PyTorch [`TransformerEncoder`](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html)
- PyTorch [Autograd tutorial](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)
- Dataset source: [Tiny Shakespeare](https://github.com/karpathy/char-rnn/tree/master/data/tinyshakespeare)

**Completion check:** explain tokens, `(B,T,C)`, causal leakage, logits, loss,
backpropagation, validation, and temperature in your own words. If any explanation
is fuzzy, rerun its smallest visualization or shape check.
